# HDBSCAN Extra Analysis

This notebook is for exploratory analysis on top of the HDBSCAN asset impact pipeline outputs.

It does not rebuild the pipeline tables. It assumes the HDBSCAN pipeline notebook has already been run and that the DuckDB tables exist.

In [1]:
import duckdb
import pandas as pd

from IPython.display import Markdown, display

DB_PATH = 'developer_project.duckdb'
con = duckdb.connect(DB_PATH)

CLUSTER_PROFILE_TABLE = 'cluster_profile_asset_base_v2'
CLUSTER_ASSET_PRIORITY_TABLE = 'cluster_asset_priority_by_cluster_v2'
CLUSTER_PERSONA_PROFILE_TABLE = 'cluster_persona_profile_v2'
CLUSTER_JOURNEY_PROFILE_TABLE = 'cluster_journey_profile_v2'
CLUSTER_EFFORT_PROFILE_TABLE = 'cluster_effort_profile_v2'
CLUSTER_TOP_PERSONA_SUMMARY_TABLE = 'cluster_top_persona_summary_v2'
CLUSTER_TOP_ASSET_SUMMARY_TABLE = 'cluster_top_asset_summary_v2'
CLUSTER_CORRELATION_SUMMARY_TABLE = 'cluster_correlation_summary_v2'
CLUSTER_GROUP_ROLLUP_TABLE = 'cluster_group_rollup_summary_v2'
CLUSTER_GROUP_ASSET_PROFILE_TABLE = 'cluster_group_asset_profile_v2'
CLUSTER_GROUP_COMPOSITION_TABLE = 'cluster_group_composition_summary_v2'
ASSET_AUDIENCE_TABLE = 'asset_audience_summary_v2'


In [2]:
required_tables = [
    CLUSTER_PROFILE_TABLE,
    CLUSTER_ASSET_PRIORITY_TABLE,
    CLUSTER_PERSONA_PROFILE_TABLE,
    CLUSTER_JOURNEY_PROFILE_TABLE,
    CLUSTER_EFFORT_PROFILE_TABLE,
    CLUSTER_TOP_PERSONA_SUMMARY_TABLE,
    CLUSTER_TOP_ASSET_SUMMARY_TABLE,
    CLUSTER_CORRELATION_SUMMARY_TABLE,
    CLUSTER_GROUP_ROLLUP_TABLE,
    CLUSTER_GROUP_ASSET_PROFILE_TABLE,
    CLUSTER_GROUP_COMPOSITION_TABLE,
    ASSET_AUDIENCE_TABLE,
]

missing = []
for table_name in required_tables:
    try:
        con.execute(f'SELECT 1 FROM {table_name} LIMIT 1')
    except Exception:
        missing.append(table_name)

if missing:
    raise ValueError(f'Missing pipeline output tables: {missing}')

display(Markdown('### Pipeline Tables Available'))
display(pd.DataFrame({'table_name': required_tables}))

### Pipeline Tables Available

,table_name
0,cluster_profile_asset_base_v2
1,cluster_asset_priority_by_cluster_v2
2,cluster_persona_profile_v2
3,cluster_journey_profile_v2
4,cluster_effort_profile_v2
5,cluster_top_persona_summary_v2
6,cluster_top_asset_summary_v2
7,cluster_correlation_summary_v2
8,cluster_group_rollup_summary_v2
9,cluster_group_asset_profile_v2


## Lifecycle Group Analysis

These views keep the analysis at the `active`, `cooling`, and `at_risk` level.

In [3]:
display(Markdown('### Lifecycle Group Rollup Summary'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_GROUP_ROLLUP_TABLE}
ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END
""").fetchdf())

display(Markdown('### Lifecycle Group Asset Profile'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_GROUP_ASSET_PROFILE_TABLE}
ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END
""").fetchdf())

display(Markdown('### Lifecycle Group Composition Table'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_GROUP_COMPOSITION_TABLE}
ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END
""").fetchdf())

### Lifecycle Group Rollup Summary

,cluster_group,developers,top_persona_1,top_persona_1_share,top_persona_2,top_persona_2_share,top_persona_3,top_persona_3_share,top_effort_1,top_effort_1_share,top_effort_2,top_effort_2_share,top_volume_asset_1,top_volume_asset_2,top_volume_asset_3,top_breadth_asset_1,top_intensity_asset_1
0,active,418049,GenAI,0.698562,CUDA,0.138912,Robotics,0.060670,very high effort,0.962586,low effort,0.031159,ngc_download,devzone_download,forum_contribution,devzone_download,ngc_download
1,cooling,356500,GenAI,0.434056,CUDA,0.285316,Robotics,0.094003,very high effort,0.535352,high effort,0.390418,devzone_download,ngc_download,dli_training,devzone_download,ngc_download
2,at_risk,1580877,CUDA,0.408582,GenAI,0.340516,Robotics,0.082461,low effort,0.387062,medium effort,0.307890,devzone_download,ngc_download,dli_training,devzone_download,ngc_download


### Lifecycle Group Asset Profile

,cluster_group,developers,pct_dli_training,lift_dli_training,intensity_dli_training,pct_webinar,lift_webinar,intensity_webinar,pct_forum_contribution,lift_forum_contribution,...,intensity_bug_filed,pct_hackathon,lift_hackathon,intensity_hackathon,pct_devzone_download,lift_devzone_download,intensity_devzone_download,pct_ngc_download,lift_ngc_download,intensity_ngc_download
0,active,418049,0.086629,0.733580,2.564904,0.016890,0.822280,2.366520,0.010542,1.264678,...,30.040752,0.000033,0.268460,1.071429,0.174097,0.412078,40.791566,0.017670,1.314477,732.196426
1,cooling,356500,0.183868,1.557015,2.034570,0.042275,2.058084,1.847787,0.013966,1.675509,...,19.906067,0.000070,0.562160,1.280000,0.339518,0.803620,25.668286,0.018463,1.373443,74.474020
2,at_risk,1580877,0.243432,2.061408,1.700735,0.037057,1.804042,1.609914,0.010389,1.246288,...,20.974576,0.000083,0.669353,1.098485,0.409577,0.969447,14.468510,0.023156,1.722529,26.366361


### Lifecycle Group Composition Table

,cluster_group,developers,top_persona_1,top_persona_1_share,top_persona_2,top_persona_2_share,top_persona_3,top_persona_3_share,top_journey_1,top_journey_1_share,top_journey_2,top_journey_2_share,top_effort_1,top_effort_1_share,top_effort_2,top_effort_2_share
0,active,418049,GenAI,0.698562,CUDA,0.138912,Robotics,0.060670,Evaluator,0.671924,Learner,0.204490,very high effort,0.962586,low effort,0.031159
1,cooling,356500,GenAI,0.434056,CUDA,0.285316,Robotics,0.094003,Historically_Active,0.929719,Builder,0.068519,very high effort,0.535352,high effort,0.390418
2,at_risk,1580877,CUDA,0.408582,GenAI,0.340516,Robotics,0.082461,Historically_Active,0.948792,Builder,0.048848,low effort,0.387062,medium effort,0.307890


## Cluster Analysis

These views stay at the per-cluster level so the lifecycle group summaries can be tied back to concrete cluster patterns.

In [4]:
display(Markdown('### Top Personas By Cluster'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_TOP_PERSONA_SUMMARY_TABLE}
ORDER BY cluster_label
""").fetchdf())

display(Markdown('### Top Assets By Cluster'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_TOP_ASSET_SUMMARY_TABLE}
ORDER BY cluster_label
""").fetchdf())

display(Markdown('### Cluster Correlation Summary'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_CORRELATION_SUMMARY_TABLE}
ORDER BY cluster_group, cluster_developers DESC, cluster_label
""").fetchdf())

### Top Personas By Cluster

,cluster_label,top_persona_1,top_persona_1_share,top_persona_2,top_persona_2_share,top_persona_3,top_persona_3_share
0,Dormant_Former_Builders,CUDA,0.686126,Simulation,0.105529,Unknown,0.075067
1,Dormant_Low_Depth,CUDA,0.529681,GenAI,0.193532,Learning_Community,0.098306
2,Dormant_One_Time_Users,GenAI,0.371202,CUDA,0.230721,Unknown,0.192822
3,active_0,CUDA,0.583847,GenAI,0.153483,Unknown,0.115459
4,active_1,GenAI,0.957485,CUDA,0.018191,Unknown,0.016496
5,active_2,GenAI,0.460621,Learning_Community,0.197056,CUDA,0.167532
6,active_3,GenAI,0.975212,Unknown,0.022816,CUDA,0.000850
7,active_4,GenAI,0.832444,Robotics,0.081244,CUDA,0.059895
8,active_5,GenAI,0.845376,Unknown,0.104506,Robotics,0.019125
9,active_noise,CUDA,0.376407,GenAI,0.367827,Robotics,0.162012


### Top Assets By Cluster

,cluster_label,top_volume_asset_1,top_volume_asset_2,top_volume_asset_3,top_breadth_asset_1,top_breadth_asset_2,top_breadth_asset_3,top_intensity_asset_1,top_intensity_asset_2,top_intensity_asset_3
0,Dormant_Former_Builders,devzone_download,ngc_download,forum_contribution,devzone_download,ngc_download,dli_training,ngc_download,devzone_download,forum_contribution
1,Dormant_Low_Depth,devzone_download,dli_training,forum_contribution,devzone_download,dli_training,webinar,bug_filed,forum_contribution,ngc_download
2,Dormant_One_Time_Users,webinar,devzone_download,dli_training,webinar,devzone_download,dli_training,webinar,devzone_download,dli_training
3,active_0,devzone_download,ngc_download,webinar,devzone_download,ngc_download,webinar,ngc_download,devzone_download,webinar
4,active_1,ngc_download,webinar,devzone_download,ngc_download,webinar,devzone_download,ngc_download,webinar,devzone_download
5,active_2,dli_training,devzone_download,forum_contribution,dli_training,devzone_download,forum_contribution,bug_filed,forum_contribution,devzone_download
6,active_3,forum_contribution,ngc_download,bug_filed,forum_contribution,ngc_download,bug_filed,forum_contribution,ngc_download,bug_filed
7,active_4,devzone_download,webinar,forum_contribution,devzone_download,webinar,forum_contribution,devzone_download,webinar,forum_contribution
8,active_5,webinar,dli_training,devzone_download,webinar,dli_training,devzone_download,bug_filed,devzone_download,dli_training
9,active_noise,ngc_download,devzone_download,forum_contribution,devzone_download,dli_training,ngc_download,ngc_download,forum_contribution,devzone_download


### Cluster Correlation Summary

,cluster_label,cluster_group,cluster_developers,top_persona_1,top_persona_1_share,top_persona_2,top_persona_2_share,top_persona_3,top_persona_3_share,dominant_effort,...,top_intensity_asset_2,top_intensity_asset_3,dominant_segment_persona,dominant_segment_effort,dominant_segment_journey,dominant_segment_developers,dominant_segment_share,dominant_segment_top_volume_asset,dominant_segment_top_breadth_asset,dominant_segment_top_intensity_asset
0,active_5,active,155034,GenAI,0.845376,Unknown,0.104506,Robotics,0.019125,very high effort,...,devzone_download,dli_training,GenAI,very high effort,Evaluator,130352,1.0,webinar,webinar,bug_filed
1,active_noise,active,105702,CUDA,0.376407,GenAI,0.367827,Robotics,0.162012,very high effort,...,forum_contribution,devzone_download,GenAI,very high effort,Learner,19543,1.0,dli_training,dli_training,devzone_download
2,active_1,active,66682,GenAI,0.957485,CUDA,0.018191,Unknown,0.016496,very high effort,...,webinar,devzone_download,GenAI,very high effort,Evaluator,63832,1.0,ngc_download,ngc_download,ngc_download
3,active_3,active,29409,GenAI,0.975212,Unknown,0.022816,CUDA,0.000850,very high effort,...,ngc_download,bug_filed,GenAI,very high effort,Learner,28677,1.0,forum_contribution,forum_contribution,forum_contribution
4,active_2,active,24658,GenAI,0.460621,Learning_Community,0.197056,CUDA,0.167532,very high effort,...,forum_contribution,devzone_download,GenAI,very high effort,Evaluator,11349,1.0,dli_training,dli_training,bug_filed
5,active_4,active,18549,GenAI,0.832444,Robotics,0.081244,CUDA,0.059895,very high effort,...,webinar,forum_contribution,Robotics,very high effort,Evaluator,1507,1.0,forum_contribution,forum_contribution,forum_contribution
6,active_0,active,18015,CUDA,0.583847,GenAI,0.153483,Unknown,0.115459,very high effort,...,devzone_download,webinar,CUDA,very high effort,Builder,10518,1.0,devzone_download,devzone_download,ngc_download
7,at_risk_0,at_risk,436825,CUDA,0.387109,GenAI,0.342382,Robotics,0.096462,high effort,...,devzone_download,forum_contribution,CUDA,high effort,Historically_Active,84553,1.0,devzone_download,devzone_download,forum_contribution
8,at_risk_5,at_risk,366741,CUDA,0.552003,GenAI,0.201047,Robotics,0.131842,medium effort,...,ngc_download,forum_contribution,CUDA,medium effort,Historically_Active,113233,1.0,devzone_download,devzone_download,devzone_download
9,at_risk_2,at_risk,210376,GenAI,0.584515,CUDA,0.168674,Learning_Community,0.102726,medium effort,...,dli_training,webinar,GenAI,medium effort,Historically_Active,110334,1.0,dli_training,dli_training,dli_training


In [5]:
display(Markdown('### Cluster Comparison View'))
display(con.execute(f"""
SELECT
    cluster_label,
    cluster_group,
    cluster_developers,
    top_persona_1,
    top_persona_1_share,
    dominant_journey,
    dominant_journey_share,
    dominant_effort,
    dominant_effort_share,
    top_volume_asset_1,
    top_volume_asset_2,
    top_breadth_asset_1,
    top_intensity_asset_1
FROM {CLUSTER_CORRELATION_SUMMARY_TABLE}
ORDER BY cluster_group, cluster_developers DESC, cluster_label
""").fetchdf())

### Cluster Comparison View

,cluster_label,cluster_group,cluster_developers,top_persona_1,top_persona_1_share,dominant_journey,dominant_journey_share,dominant_effort,dominant_effort_share,top_volume_asset_1,top_volume_asset_2,top_breadth_asset_1,top_intensity_asset_1
0,active_5,active,155034,GenAI,0.845376,Evaluator,0.984848,very high effort,1.000000,webinar,dli_training,webinar,bug_filed
1,active_noise,active,105702,CUDA,0.376407,Learner,0.508259,very high effort,0.852103,ngc_download,devzone_download,devzone_download,ngc_download
2,active_1,active,66682,GenAI,0.957485,Evaluator,0.999475,very high effort,0.999880,ngc_download,webinar,ngc_download,ngc_download
3,active_3,active,29409,GenAI,0.975212,Learner,0.999558,very high effort,1.000000,forum_contribution,ngc_download,forum_contribution,forum_contribution
4,active_2,active,24658,GenAI,0.460621,Evaluator,0.997769,very high effort,1.000000,dli_training,devzone_download,dli_training,bug_filed
5,active_4,active,18549,GenAI,0.832444,Evaluator,0.999030,very high effort,1.000000,devzone_download,webinar,devzone_download,devzone_download
6,active_0,active,18015,CUDA,0.583847,Builder,1.000000,very high effort,1.000000,devzone_download,ngc_download,devzone_download,ngc_download
7,at_risk_0,at_risk,436825,CUDA,0.387109,Historically_Active,0.944740,high effort,0.436747,devzone_download,dli_training,devzone_download,ngc_download
8,at_risk_5,at_risk,366741,CUDA,0.552003,Historically_Active,0.897025,medium effort,0.522562,devzone_download,dli_training,devzone_download,devzone_download
9,at_risk_2,at_risk,210376,GenAI,0.584515,Historically_Active,1.000000,medium effort,0.760168,dli_training,devzone_download,dli_training,devzone_download


## Asset-Centered Analysis

This flips the direction of the analysis from `group -> assets` to `asset -> audience`.

In [6]:
display(Markdown('### Asset Audience Table'))
display(con.execute(f"""
SELECT *
FROM {ASSET_AUDIENCE_TABLE}
ORDER BY exposed_developers DESC, asset_name
""").fetchdf())

display(Markdown('### Asset Audience: Sorted By Active Share'))
display(con.execute(f"""
SELECT *
FROM {ASSET_AUDIENCE_TABLE}
ORDER BY pct_active DESC, exposed_developers DESC, asset_name
""").fetchdf())

display(Markdown('### Asset Audience: Sorted By Learner Share'))
display(con.execute(f"""
SELECT *
FROM {ASSET_AUDIENCE_TABLE}
ORDER BY pct_learner DESC, exposed_developers DESC, asset_name
""").fetchdf())

### Asset Audience Table

,asset_name,asset_group,asset_role,exposed_developers,pct_active,pct_cooling,pct_at_risk,pct_builder,pct_learner,pct_high_effort,top_persona
0,devzone_download,activity_asset,download_activity,3962571,0.018367,0.030545,0.163402,0.077534,0.004258,0.248460,CUDA
1,dli_training,activity_asset,learning_activity,1107590,0.032697,0.059182,0.347453,0.029501,0.007366,0.404712,GenAI
2,webinar,activity_asset,learning_activity,192657,0.036651,0.078227,0.304074,0.072004,0.016449,0.441816,GenAI
3,ngc_download,activity_asset,download_activity,126082,0.058589,0.052204,0.290335,0.317857,0.006163,0.574539,GenAI
4,forum_contribution,activity_asset,community_activity,78181,0.056369,0.063686,0.210064,0.329978,0.014991,0.658894,CUDA
5,bug_filed,activity_asset,community_activity,8195,0.077852,0.062355,0.302379,0.259671,0.006711,0.738133,CUDA
6,hackathon,activity_asset,community_activity,1170,0.011966,0.021368,0.112821,0.128205,0.002564,0.271795,CUDA


### Asset Audience: Sorted By Active Share

,asset_name,asset_group,asset_role,exposed_developers,pct_active,pct_cooling,pct_at_risk,pct_builder,pct_learner,pct_high_effort,top_persona
0,bug_filed,activity_asset,community_activity,8195,0.077852,0.062355,0.302379,0.259671,0.006711,0.738133,CUDA
1,ngc_download,activity_asset,download_activity,126082,0.058589,0.052204,0.290335,0.317857,0.006163,0.574539,GenAI
2,forum_contribution,activity_asset,community_activity,78181,0.056369,0.063686,0.210064,0.329978,0.014991,0.658894,CUDA
3,webinar,activity_asset,learning_activity,192657,0.036651,0.078227,0.304074,0.072004,0.016449,0.441816,GenAI
4,dli_training,activity_asset,learning_activity,1107590,0.032697,0.059182,0.347453,0.029501,0.007366,0.404712,GenAI
5,devzone_download,activity_asset,download_activity,3962571,0.018367,0.030545,0.163402,0.077534,0.004258,0.248460,CUDA
6,hackathon,activity_asset,community_activity,1170,0.011966,0.021368,0.112821,0.128205,0.002564,0.271795,CUDA


### Asset Audience: Sorted By Learner Share

,asset_name,asset_group,asset_role,exposed_developers,pct_active,pct_cooling,pct_at_risk,pct_builder,pct_learner,pct_high_effort,top_persona
0,webinar,activity_asset,learning_activity,192657,0.036651,0.078227,0.304074,0.072004,0.016449,0.441816,GenAI
1,forum_contribution,activity_asset,community_activity,78181,0.056369,0.063686,0.210064,0.329978,0.014991,0.658894,CUDA
2,dli_training,activity_asset,learning_activity,1107590,0.032697,0.059182,0.347453,0.029501,0.007366,0.404712,GenAI
3,bug_filed,activity_asset,community_activity,8195,0.077852,0.062355,0.302379,0.259671,0.006711,0.738133,CUDA
4,ngc_download,activity_asset,download_activity,126082,0.058589,0.052204,0.290335,0.317857,0.006163,0.574539,GenAI
5,devzone_download,activity_asset,download_activity,3962571,0.018367,0.030545,0.163402,0.077534,0.004258,0.248460,CUDA
6,hackathon,activity_asset,community_activity,1170,0.011966,0.021368,0.112821,0.128205,0.002564,0.271795,CUDA


## Overall Journey Context

This is useful when the lifecycle groups or asset views appear to be dominated by historically active users.

In [7]:
print('### Overall Journey Stage Distribution')
print(con.execute(f"""
SELECT
    behavior_journey_stage_30d AS journey_stage,
    COUNT(*) AS developers,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () AS pct_of_all_developers
FROM {CLUSTER_PROFILE_TABLE}
GROUP BY 1
ORDER BY developers DESC
""").fetchdf())

print('\n### Overall Current Journey State Distribution')
print(con.execute(f"""
SELECT
    current_journey_state_30d AS current_journey_state,
    COUNT(*) AS developers,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () AS pct_of_all_developers
FROM {CLUSTER_PROFILE_TABLE}
GROUP BY 1
ORDER BY developers DESC
""").fetchdf())

print('\n### Overall Final Lifecycle Status Distribution')
print(con.execute(f"""
SELECT
    final_lifecycle_status,
    COUNT(*) AS developers,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () AS pct_of_all_developers
FROM {CLUSTER_PROFILE_TABLE}
GROUP BY 1
ORDER BY developers DESC
""").fetchdf())

### Overall Journey Stage Distribution
         journey_stage  developers  pct_of_all_developers
0  Historically_Active     6963992               0.742494
1          Unactivated     1718912               0.183269
2              Builder      319991               0.034117
3            Evaluator      289432               0.030859
4              Learner       85487               0.009115
5             Explorer        1376               0.000147

### Overall Current Journey State Distribution
          current_journey_state  developers  pct_of_all_developers
0   Dormant_Historically_Active     5132624               0.547235
1                   Unactivated     1718912               0.183269
2   At_Risk_Historically_Active     1499923               0.159920
3   Cooling_Historically_Active      331445               0.035338
4                     Evaluator      280897               0.029949
5               Dormant_Builder      168052               0.017918
6                       Learner       

## Notes

- Keep adding exploratory tables here first.
- Once a table proves useful, decide later whether it belongs back in the pipeline.
- This notebook is the safer place for long-form interpretation and recommendation writing.

In [8]:
con.close()